# Lab 2 — Data Preprocessing and Exploration

**Course:** Data Mining Lab  
**Notebook:** Lab 2 — Data Preprocessing and Exploration

## Objectives
- Load a dataset, inspect structure, handle missing values and duplicates.
- Encode categorical variables, normalize/standardize numeric features, and summarize data.

> Run cells from top to bottom. The notebook uses a reproducible random seed. For Adult Income labs, it attempts to load the public UCI Adult dataset and uses a clearly labelled local synthetic fallback if the download is unavailable.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
COLUMNS = ["age","workclass","fnlwgt","education","education.num","marital.status",
           "occupation","relationship","race","sex","capital.gain","capital.loss",
           "hours.per.week","native.country","income"]

def load_adult():
    local = Path("adult.csv")
    if local.exists():
        data = pd.read_csv(local)
        if set(COLUMNS).issubset(data.columns):
            return data[COLUMNS].copy(), "local adult.csv"
    try:
        data = pd.read_csv(DATA_URL, names=COLUMNS, na_values="?", skipinitialspace=True)
        return data, "UCI Adult dataset (downloaded)"
    except Exception as exc:
        rng = np.random.default_rng(42)
        n = 1200
        age = rng.integers(18, 75, n)
        hours = np.clip(rng.normal(40, 12, n).round(), 1, 80).astype(int)
        edu = rng.integers(1, 16, n)
        gain = rng.choice([0,0,0,0,500,1500,5000,10000], n)
        score = -5 + 0.055*age + 0.10*edu + 0.025*hours + (gain>0)*1.0
        income = np.where(rng.random(n) < 1/(1+np.exp(-score)), ">50K", "<=50K")
        data = pd.DataFrame({
            "age":age,"workclass":rng.choice(["Private","Self-emp","Government"],n),
            "fnlwgt":rng.integers(20000,500000,n),
            "education":rng.choice(["HS-grad","Bachelors","Masters","Some-college"],n),
            "education.num":edu,"marital.status":rng.choice(["Married","Never-married","Divorced"],n),
            "occupation":rng.choice(["Tech","Sales","Service","Admin"],n),
            "relationship":rng.choice(["Husband","Not-in-family","Own-child"],n),
            "race":rng.choice(["White","Black","Other"],n),"sex":rng.choice(["Male","Female"],n),
            "capital.gain":gain,"capital.loss":rng.choice([0,0,0,500,1000],n),
            "hours.per.week":hours,"native.country":rng.choice(["United-States","Other"],n),
            "income":income})
        print("NOTICE: Public dataset unavailable; using reproducible synthetic teaching data. Reason:", str(exc)[:180])
        return data, "synthetic fallback (not UCI observations)"

df, DATA_SOURCE = load_adult()
print("Data source:", DATA_SOURCE, "| Shape:", df.shape)
display(df.head())

print("\nData types:")
print(df.dtypes)
print("\nMissing values before cleaning:")
print(df.isna().sum())

## Handle missing values and duplicates

In [ ]:
clean = df.copy()
clean.replace("?", np.nan, inplace=True)
for col in clean.columns:
    if clean[col].isna().any():
        if clean[col].dtype == "object":
            clean[col] = clean[col].fillna(clean[col].mode().iloc[0])
        else:
            clean[col] = clean[col].fillna(clean[col].median())
before = len(clean)
clean = clean.drop_duplicates().reset_index(drop=True)
print("Duplicate rows removed:", before-len(clean))
print("Remaining missing values:", int(clean.isna().sum().sum()))

## Encode categories and scale numerical features

In [ ]:
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler
numeric_cols = clean.select_dtypes(include=np.number).columns.tolist()
categorical_cols = clean.select_dtypes(exclude=np.number).columns.tolist()
encoded = pd.get_dummies(clean, columns=categorical_cols, drop_first=True, dtype=int)
print("Encoded shape:", encoded.shape)
minmax = MinMaxScaler()
standard = StandardScaler()
normalized = pd.DataFrame(minmax.fit_transform(clean[numeric_cols]), columns=numeric_cols)
standardized = pd.DataFrame(standard.fit_transform(clean[numeric_cols]), columns=numeric_cols)
display(normalized.head())
display(standardized.head())

## Descriptive statistics

In [ ]:
display(clean.describe(include="all").T)